# Demo — Atención en un modelo real: cada cabeza hace algo distinto

> Segunda parte de [la demo de atención](atencion.qmd), que acompaña a
> [Arquitectura
> Transformer](../bloque1/01a-transformer.qmd#sec-atencion-multi-cabeza).
> Se ejecuta en el navegador con Shinylive y no necesita clave de API.

> **El modelo es un BERT, no un LLM generativo**
>
> Los pesos de esta demo son **reales**, extraídos de BETO, una versión
> de BERT entrenada en español. BERT es un *codificador*: cada palabra
> puede mirar a las que van **antes y después**. Los LLM generativos
> (ChatGPT, Qwen, Llama…) son *decodificadores* y solo miran hacia
> atrás. Aquí se usa un codificador porque sus pesos son los más fáciles
> de leer y de interpretar.

En [la primera parte](atencion.qmd) la atención se mostró con pesos
diseñados a mano y una sola cabeza. En un modelo real no hay una cabeza,
sino muchas: BETO tiene 12 capas con 12 cabezas cada una, es decir,
**144 cabezas** que trabajan a la vez, cada una con sus propios pesos.

Aquí se muestran **seis**, elegidas para enseñar tipos de comportamiento
distintos. Cada tarjeta muestra la **huella** de una cabeza: el patrón
completo de a qué atiende cada token. Pulsa una tarjeta para ampliarla y
un token para seleccionarlo; la fila naranja de todas las huellas cambia
a la vez, así que puedes comparar las seis cabezas de un vistazo.

``` shinylive-python
#| '!! shinylive warning !!': |
#|   shinylive does not work in self-contained HTML documents.
#|   Please set `embed-resources: false` in your metadata.
#| standalone: true
#| viewerHeight: 1150

from shiny import reactive
from shiny.express import input, ui, render

# DATA-START
DATA = {"model":"dccuchile/bert-base-spanish-wwm-cased","heads":[{"id":"L3H8","layer":3,"head":8,"label":"Mira a la palabra anterior","desc":"Cada token atiende casi del todo al token que tiene justo a la izquierda."},{"id":"L5H7","layer":5,"head":7,"label":"Mira a la palabra siguiente","desc":"Cada token atiende casi del todo al token que tiene justo a la derecha."},{"id":"L10H5","layer":10,"head":5,"label":"Se ancla al inicio y al final","desc":"Casi toda la atención va a [CLS] y [SEP], los marcadores de inicio y fin: una cabeza que 'descansa'."},{"id":"L6H10","layer":6,"head":10,"label":"Tiende a unir sujeto y verbo","desc":"En estas frases el sujeto atiende al verbo de su oración, aunque haya palabras entre medias. Una parte grande de su atención va, además, a [CLS]/[SEP]."},{"id":"L1H3","layer":1,"head":3,"label":"Tiende a unir objeto y verbo","desc":"En estas frases el objeto atiende al verbo que lo rige, aunque haya palabras entre medias. Una parte grande de su atención va, además, a [CLS]/[SEP]."},{"id":"L6H8","layer":6,"head":8,"label":"Reparte sin un patrón claro","desc":"La atención se reparte entre muchos tokens sin un patrón fácil de describir."}],"sentences":[{"text":"El perro grande persiguió al gato negro .","tokens":["[CLS]","El","perro","grande","persigui","##ó","al","gato","negro",".","[SEP]"],"default":2,"att":{"L3H8":[[0.816,0.003,0.0,0.0,0.001,0.007,0.018,0.013,0.004,0.074,0.064],[0.999,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.001,0.999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.999,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.996,0.004,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.879,0.12]],"L5H7":[[0.549,0.008,0.0,0.0,0.0,0.002,0.0,0.0,0.0,0.032,0.409],[0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.999,0.001,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005,0.995,0.0],[0.034,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.077,0.888],[0.201,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.131,0.667]],"L10H5":[[0.061,0.096,0.017,0.06,0.071,0.078,0.063,0.012,0.043,0.131,0.368],[0.056,0.007,0.004,0.002,0.005,0.005,0.002,0.003,0.005,0.01,0.901],[0.138,0.003,0.023,0.004,0.004,0.004,0.001,0.019,0.03,0.032,0.741],[0.051,0.002,0.004,0.005,0.002,0.004,0.001,0.004,0.016,0.014,0.897],[0.036,0.001,0.002,0.001,0.004,0.004,0.001,0.003,0.006,0.009,0.931],[0.028,0.001,0.001,0.001,0.004,0.006,0.001,0.001,0.004,0.008,0.945],[0.028,0.009,0.011,0.003,0.014,0.013,0.012,0.012,0.015,0.012,0.871],[0.089,0.002,0.015,0.003,0.005,0.004,0.002,0.031,0.05,0.038,0.76],[0.046,0.003,0.006,0.005,0.004,0.007,0.002,0.011,0.03,0.027,0.86],[0.08,0.042,0.025,0.022,0.049,0.042,0.013,0.022,0.047,0.032,0.623],[0.009,0.001,0.002,0.001,0.002,0.004,0.0,0.001,0.003,0.004,0.972]],"L6H10":[[0.295,0.07,0.014,0.023,0.024,0.054,0.046,0.009,0.017,0.146,0.303],[0.018,0.02,0.005,0.011,0.362,0.211,0.159,0.018,0.013,0.143,0.04],[0.068,0.03,0.011,0.013,0.228,0.224,0.078,0.017,0.011,0.174,0.147],[0.065,0.032,0.014,0.014,0.1,0.158,0.079,0.032,0.025,0.222,0.259],[0.095,0.018,0.002,0.001,0.035,0.022,0.062,0.009,0.014,0.398,0.344],[0.062,0.021,0.004,0.001,0.023,0.022,0.035,0.022,0.017,0.53,0.263],[0.066,0.031,0.008,0.001,0.032,0.014,0.029,0.026,0.013,0.515,0.264],[0.136,0.032,0.015,0.002,0.009,0.007,0.01,0.014,0.009,0.27,0.494],[0.177,0.029,0.016,0.002,0.006,0.007,0.012,0.016,0.004,0.22,0.512],[0.188,0.051,0.008,0.003,0.03,0.038,0.042,0.006,0.005,0.202,0.427],[0.147,0.045,0.007,0.003,0.035,0.069,0.024,0.006,0.007,0.127,0.529]],"L1H3":[[0.964,0.006,0.001,0.002,0.002,0.003,0.004,0.002,0.001,0.008,0.009],[0.987,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.001],[0.883,0.091,0.007,0.004,0.003,0.001,0.0,0.002,0.001,0.004,0.004],[0.06,0.723,0.131,0.027,0.022,0.004,0.003,0.002,0.001,0.014,0.013],[0.681,0.13,0.066,0.054,0.012,0.042,0.002,0.002,0.001,0.004,0.007],[0.324,0.34,0.061,0.076,0.08,0.04,0.054,0.001,0.0,0.007,0.018],[0.158,0.012,0.01,0.03,0.622,0.128,0.035,0.002,0.0,0.001,0.002],[0.616,0.009,0.009,0.018,0.152,0.074,0.082,0.019,0.009,0.007,0.007],[0.167,0.008,0.011,0.006,0.02,0.04,0.33,0.34,0.033,0.035,0.01],[0.378,0.025,0.004,0.004,0.014,0.024,0.182,0.062,0.067,0.167,0.073],[0.936,0.002,0.0,0.0,0.001,0.001,0.003,0.002,0.002,0.032,0.021]],"L6H8":[[0.021,0.117,0.139,0.056,0.107,0.092,0.056,0.139,0.106,0.12,0.046],[0.044,0.113,0.128,0.169,0.041,0.085,0.173,0.132,0.046,0.054,0.015],[0.238,0.065,0.063,0.12,0.028,0.047,0.074,0.067,0.094,0.127,0.076],[0.128,0.078,0.123,0.1,0.013,0.041,0.061,0.099,0.079,0.128,0.15],[0.156,0.079,0.035,0.041,0.022,0.069,0.107,0.037,0.052,0.103,0.298],[0.062,0.225,0.15,0.142,0.045,0.071,0.132,0.076,0.051,0.036,0.011],[0.12,0.184,0.147,0.111,0.042,0.088,0.087,0.079,0.058,0.047,0.038],[0.409,0.06,0.054,0.071,0.02,0.03,0.036,0.035,0.091,0.1,0.093],[0.182,0.062,0.1,0.088,0.051,0.05,0.034,0.085,0.048,0.138,0.164],[0.089,0.098,0.189,0.109,0.071,0.143,0.024,0.088,0.062,0.107,0.02],[0.2,0.028,0.032,0.018,0.015,0.034,0.015,0.034,0.027,0.109,0.487]]}},{"text":"La profesora explicó la lección difícil a los alumnos .","tokens":["[CLS]","La","profesora","explicó","la","lección","difícil","a","los","alumnos",".","[SEP]"],"default":2,"att":{"L3H8":[[0.881,0.009,0.0,0.0,0.0,0.001,0.004,0.015,0.003,0.003,0.016,0.066],[1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.001,0.999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.992,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.998,0.002,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.998,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.996,0.004,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006,0.884,0.109]],"L5H7":[[0.774,0.008,0.008,0.005,0.001,0.0,0.0,0.0,0.0,0.004,0.014,0.185],[0.0,0.007,0.993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.001,0.999,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.999,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.999,0.0],[0.049,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.045,0.905],[0.076,0.0,0.0,0.0,0.01,0.002,0.0,0.0,0.0,0.0,0.068,0.844]],"L10H5":[[0.027,0.047,0.042,0.09,0.08,0.047,0.058,0.024,0.047,0.054,0.096,0.387],[0.042,0.006,0.01,0.011,0.004,0.018,0.02,0.001,0.001,0.011,0.008,0.869],[0.097,0.005,0.011,0.008,0.004,0.028,0.015,0.001,0.001,0.009,0.014,0.809],[0.02,0.001,0.002,0.005,0.002,0.012,0.011,0.0,0.001,0.003,0.006,0.935],[0.088,0.006,0.009,0.023,0.015,0.035,0.037,0.002,0.004,0.012,0.017,0.753],[0.059,0.003,0.019,0.01,0.011,0.085,0.036,0.003,0.003,0.02,0.01,0.742],[0.042,0.023,0.023,0.032,0.044,0.084,0.045,0.011,0.017,0.032,0.018,0.628],[0.022,0.006,0.011,0.009,0.005,0.011,0.011,0.009,0.005,0.013,0.018,0.881],[0.017,0.006,0.02,0.012,0.004,0.022,0.013,0.003,0.005,0.028,0.009,0.861],[0.173,0.008,0.026,0.015,0.004,0.039,0.015,0.001,0.001,0.015,0.024,0.677],[0.044,0.01,0.034,0.017,0.01,0.067,0.034,0.003,0.005,0.018,0.037,0.72],[0.009,0.001,0.003,0.003,0.001,0.003,0.003,0.0,0.0,0.002,0.002,0.973]],"L6H10":[[0.201,0.042,0.026,0.15,0.074,0.067,0.03,0.021,0.028,0.06,0.092,0.21],[0.031,0.012,0.004,0.437,0.051,0.018,0.035,0.218,0.041,0.013,0.057,0.084],[0.104,0.013,0.008,0.297,0.04,0.024,0.027,0.095,0.021,0.016,0.065,0.289],[0.036,0.005,0.001,0.039,0.013,0.005,0.029,0.495,0.044,0.018,0.189,0.126],[0.021,0.002,0.001,0.018,0.034,0.022,0.058,0.499,0.061,0.015,0.145,0.124],[0.08,0.007,0.006,0.037,0.039,0.033,0.024,0.153,0.086,0.056,0.104,0.374],[0.072,0.01,0.004,0.024,0.036,0.012,0.02,0.044,0.069,0.044,0.244,0.421],[0.05,0.008,0.003,0.054,0.022,0.011,0.017,0.048,0.038,0.045,0.263,0.443],[0.042,0.005,0.002,0.012,0.019,0.017,0.007,0.017,0.054,0.071,0.277,0.476],[0.111,0.011,0.005,0.006,0.023,0.012,0.004,0.006,0.036,0.048,0.165,0.574],[0.184,0.016,0.006,0.074,0.066,0.026,0.021,0.05,0.021,0.019,0.09,0.428],[0.157,0.024,0.007,0.104,0.029,0.014,0.014,0.063,0.022,0.014,0.101,0.451]],"L1H3":[[0.946,0.007,0.006,0.005,0.004,0.001,0.001,0.005,0.006,0.001,0.009,0.009],[0.961,0.03,0.003,0.002,0.0,0.0,0.0,0.001,0.001,0.0,0.001,0.001],[0.953,0.034,0.006,0.003,0.0,0.0,0.0,0.0,0.001,0.0,0.001,0.001],[0.548,0.38,0.05,0.006,0.005,0.0,0.001,0.002,0.001,0.0,0.003,0.005],[0.122,0.186,0.143,0.502,0.031,0.007,0.0,0.0,0.001,0.001,0.003,0.005],[0.26,0.11,0.069,0.355,0.146,0.022,0.014,0.002,0.003,0.001,0.008,0.01],[0.162,0.053,0.007,0.178,0.494,0.078,0.008,0.005,0.005,0.001,0.003,0.006],[0.167,0.03,0.004,0.103,0.325,0.125,0.146,0.08,0.008,0.001,0.003,0.007],[0.18,0.021,0.006,0.057,0.049,0.234,0.243,0.117,0.062,0.013,0.008,0.009],[0.399,0.057,0.007,0.006,0.042,0.029,0.058,0.183,0.152,0.014,0.037,0.016],[0.31,0.018,0.009,0.012,0.005,0.022,0.023,0.122,0.251,0.06,0.11,0.058],[0.92,0.002,0.001,0.002,0.0,0.001,0.001,0.004,0.013,0.002,0.03,0.025]],"L6H8":[[0.02,0.053,0.088,0.155,0.044,0.105,0.085,0.043,0.038,0.144,0.089,0.136],[0.07,0.036,0.08,0.097,0.058,0.079,0.112,0.029,0.055,0.232,0.071,0.081],[0.085,0.08,0.086,0.122,0.053,0.06,0.036,0.022,0.044,0.085,0.124,0.202],[0.078,0.041,0.112,0.096,0.065,0.064,0.126,0.03,0.088,0.19,0.053,0.056],[0.059,0.024,0.046,0.103,0.079,0.053,0.171,0.066,0.119,0.181,0.055,0.046],[0.184,0.069,0.088,0.061,0.045,0.022,0.054,0.032,0.09,0.088,0.153,0.114],[0.096,0.045,0.053,0.128,0.055,0.041,0.043,0.054,0.119,0.155,0.135,0.077],[0.118,0.056,0.057,0.125,0.078,0.072,0.11,0.04,0.105,0.117,0.031,0.091],[0.127,0.076,0.072,0.125,0.051,0.087,0.097,0.023,0.071,0.174,0.042,0.055],[0.147,0.146,0.081,0.128,0.033,0.054,0.023,0.011,0.026,0.039,0.127,0.187],[0.096,0.042,0.125,0.167,0.032,0.088,0.098,0.014,0.024,0.145,0.081,0.087],[0.142,0.012,0.017,0.039,0.019,0.019,0.027,0.008,0.012,0.031,0.06,0.614]]}},{"text":"Los alumnos que estudiaron mucho aprobaron el examen final .","tokens":["[CLS]","Los","alumnos","que","estudiar","##on","mucho","aprobaron","el","examen","final",".","[SEP]"],"default":2,"att":{"L3H8":[[0.729,0.001,0.0,0.0,0.0,0.0,0.003,0.009,0.001,0.0,0.0,0.021,0.233],[0.999,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.003,0.996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.999,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.999,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.999,0.001,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.998,0.001,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.878,0.122]],"L5H7":[[0.716,0.011,0.018,0.001,0.001,0.0,0.0,0.0,0.001,0.001,0.0,0.01,0.239],[0.0,0.002,0.998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.001,0.999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.014,0.98,0.005,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004,0.996,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0],[0.021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.013,0.965],[0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.032,0.917]],"L10H5":[[0.032,0.066,0.047,0.077,0.051,0.069,0.053,0.082,0.035,0.03,0.024,0.135,0.299],[0.048,0.006,0.023,0.002,0.008,0.005,0.011,0.013,0.001,0.003,0.002,0.013,0.865],[0.217,0.015,0.045,0.002,0.006,0.004,0.007,0.018,0.002,0.008,0.009,0.086,0.583],[0.055,0.008,0.016,0.003,0.008,0.013,0.015,0.024,0.002,0.002,0.004,0.027,0.826],[0.097,0.002,0.011,0.001,0.003,0.001,0.004,0.005,0.0,0.003,0.002,0.026,0.844],[0.043,0.001,0.007,0.0,0.005,0.003,0.005,0.008,0.0,0.002,0.002,0.008,0.915],[0.029,0.005,0.017,0.003,0.013,0.004,0.023,0.008,0.004,0.005,0.005,0.008,0.875],[0.064,0.004,0.014,0.001,0.007,0.002,0.01,0.012,0.001,0.003,0.004,0.021,0.858],[0.073,0.005,0.02,0.001,0.009,0.007,0.011,0.014,0.001,0.008,0.004,0.017,0.831],[0.102,0.004,0.018,0.001,0.006,0.001,0.005,0.006,0.001,0.007,0.006,0.028,0.815],[0.072,0.005,0.031,0.001,0.009,0.005,0.01,0.012,0.002,0.012,0.008,0.017,0.814],[0.06,0.035,0.074,0.011,0.015,0.028,0.026,0.101,0.01,0.03,0.019,0.071,0.52],[0.008,0.001,0.003,0.0,0.001,0.001,0.003,0.002,0.0,0.001,0.001,0.003,0.975]],"L6H10":[[0.142,0.062,0.064,0.048,0.029,0.02,0.024,0.108,0.082,0.05,0.026,0.081,0.264],[0.009,0.016,0.003,0.006,0.003,0.001,0.014,0.84,0.035,0.004,0.001,0.048,0.02],[0.088,0.055,0.024,0.019,0.026,0.015,0.023,0.397,0.045,0.013,0.004,0.076,0.216],[0.011,0.006,0.004,0.004,0.003,0.002,0.023,0.876,0.011,0.005,0.002,0.031,0.022],[0.09,0.046,0.059,0.028,0.09,0.033,0.111,0.088,0.038,0.071,0.027,0.069,0.25],[0.052,0.041,0.022,0.009,0.014,0.01,0.034,0.494,0.042,0.024,0.01,0.103,0.144],[0.039,0.023,0.01,0.005,0.005,0.002,0.031,0.254,0.073,0.095,0.064,0.119,0.281],[0.067,0.028,0.006,0.002,0.002,0.0,0.006,0.087,0.062,0.031,0.035,0.232,0.442],[0.064,0.026,0.007,0.003,0.002,0.001,0.01,0.052,0.055,0.041,0.04,0.349,0.352],[0.115,0.018,0.013,0.002,0.002,0.0,0.003,0.003,0.011,0.025,0.014,0.149,0.644],[0.139,0.021,0.011,0.003,0.002,0.001,0.007,0.01,0.03,0.037,0.028,0.262,0.449],[0.155,0.06,0.019,0.011,0.004,0.002,0.01,0.152,0.071,0.017,0.009,0.083,0.407],[0.122,0.061,0.018,0.009,0.003,0.002,0.006,0.093,0.017,0.009,0.006,0.094,0.559]],"L1H3":[[0.947,0.01,0.002,0.006,0.002,0.005,0.001,0.002,0.005,0.001,0.001,0.008,0.01],[0.958,0.033,0.001,0.001,0.0,0.001,0.0,0.001,0.002,0.0,0.0,0.001,0.002],[0.923,0.059,0.001,0.002,0.0,0.001,0.0,0.0,0.008,0.001,0.0,0.002,0.002],[0.126,0.789,0.029,0.021,0.003,0.008,0.004,0.001,0.005,0.001,0.001,0.007,0.004],[0.456,0.147,0.157,0.052,0.019,0.139,0.006,0.003,0.001,0.001,0.005,0.009,0.005],[0.057,0.275,0.052,0.542,0.019,0.018,0.01,0.0,0.004,0.002,0.002,0.008,0.01],[0.123,0.074,0.029,0.215,0.182,0.285,0.043,0.011,0.013,0.002,0.0,0.006,0.016],[0.533,0.035,0.023,0.125,0.037,0.146,0.044,0.014,0.016,0.004,0.001,0.006,0.015],[0.177,0.013,0.002,0.011,0.056,0.11,0.179,0.378,0.063,0.004,0.001,0.002,0.004],[0.2,0.012,0.005,0.006,0.021,0.033,0.021,0.507,0.051,0.038,0.091,0.01,0.005],[0.026,0.018,0.003,0.008,0.006,0.011,0.011,0.091,0.142,0.637,0.02,0.024,0.004],[0.415,0.014,0.003,0.01,0.003,0.008,0.013,0.032,0.205,0.089,0.054,0.104,0.049],[0.909,0.002,0.0,0.001,0.0,0.001,0.0,0.002,0.006,0.003,0.005,0.038,0.033]],"L6H8":[[0.013,0.064,0.147,0.032,0.074,0.037,0.046,0.22,0.018,0.177,0.06,0.063,0.049],[0.028,0.05,0.086,0.054,0.052,0.102,0.18,0.211,0.063,0.061,0.079,0.023,0.013],[0.057,0.065,0.084,0.054,0.073,0.057,0.11,0.069,0.04,0.102,0.085,0.099,0.105],[0.058,0.07,0.139,0.072,0.137,0.07,0.125,0.1,0.044,0.042,0.072,0.04,0.03],[0.102,0.024,0.036,0.038,0.03,0.023,0.016,0.015,0.014,0.049,0.052,0.101,0.5],[0.066,0.066,0.1,0.058,0.175,0.04,0.075,0.072,0.047,0.092,0.096,0.059,0.054],[0.083,0.085,0.116,0.119,0.095,0.074,0.052,0.054,0.031,0.039,0.043,0.085,0.125],[0.033,0.121,0.111,0.267,0.052,0.082,0.071,0.039,0.044,0.028,0.039,0.04,0.071],[0.044,0.154,0.128,0.089,0.071,0.074,0.107,0.082,0.048,0.079,0.077,0.022,0.025],[0.064,0.111,0.205,0.073,0.051,0.046,0.038,0.021,0.025,0.021,0.024,0.115,0.205],[0.032,0.136,0.22,0.051,0.068,0.05,0.066,0.101,0.038,0.096,0.041,0.067,0.033],[0.082,0.059,0.237,0.04,0.063,0.083,0.072,0.124,0.012,0.098,0.03,0.076,0.024],[0.141,0.031,0.051,0.018,0.016,0.013,0.019,0.033,0.011,0.024,0.015,0.09,0.537]]}},{"text":"Mi hermana pequeña compró unos libros viejos en la librería .","tokens":["[CLS]","Mi","hermana","pequeña","compró","unos","libros","viejos","en","la","libre","##ría",".","[SEP]"],"default":2,"att":{"L3H8":[[0.551,0.001,0.0,0.0,0.0,0.001,0.003,0.007,0.005,0.001,0.0,0.001,0.078,0.353],[1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.001,0.998,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.999,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.995,0.004,0.0],[0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.835,0.164]],"L5H7":[[0.628,0.022,0.003,0.001,0.002,0.001,0.001,0.0,0.004,0.004,0.002,0.002,0.043,0.288],[0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.005,0.995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.001,0.999,0.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.999,0.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0],[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001,0.042,0.957,0.0],[0.006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.031,0.963],[0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07,0.921]],"L10H5":[[0.042,0.078,0.038,0.04,0.051,0.044,0.013,0.04,0.04,0.04,0.01,0.02,0.172,0.373],[0.076,0.003,0.001,0.001,0.003,0.001,0.001,0.001,0.0,0.0,0.0,0.001,0.01,0.901],[0.096,0.004,0.003,0.002,0.005,0.001,0.004,0.001,0.0,0.0,0.001,0.001,0.026,0.854],[0.056,0.003,0.002,0.0,0.002,0.001,0.002,0.001,0.0,0.0,0.0,0.001,0.012,0.918],[0.038,0.0,0.0,0.0,0.002,0.001,0.002,0.001,0.0,0.0,0.001,0.002,0.007,0.946],[0.102,0.003,0.001,0.001,0.007,0.003,0.006,0.003,0.001,0.001,0.002,0.006,0.019,0.844],[0.148,0.003,0.002,0.001,0.004,0.003,0.035,0.01,0.001,0.0,0.009,0.009,0.019,0.758],[0.068,0.002,0.002,0.001,0.003,0.002,0.011,0.003,0.001,0.0,0.002,0.003,0.011,0.893],[0.107,0.005,0.002,0.002,0.009,0.004,0.002,0.002,0.002,0.001,0.001,0.004,0.036,0.825],[0.064,0.007,0.002,0.002,0.006,0.003,0.003,0.002,0.001,0.001,0.001,0.003,0.024,0.882],[0.093,0.003,0.002,0.001,0.002,0.002,0.015,0.003,0.0,0.0,0.005,0.004,0.02,0.85],[0.074,0.002,0.001,0.001,0.002,0.001,0.008,0.003,0.0,0.0,0.003,0.006,0.027,0.872],[0.262,0.012,0.009,0.007,0.011,0.006,0.014,0.01,0.002,0.002,0.003,0.01,0.021,0.63],[0.016,0.002,0.001,0.001,0.005,0.001,0.002,0.001,0.0,0.0,0.001,0.001,0.006,0.962]],"L6H10":[[0.209,0.071,0.032,0.014,0.066,0.025,0.03,0.013,0.032,0.024,0.004,0.008,0.077,0.395],[0.072,0.043,0.02,0.02,0.479,0.057,0.022,0.007,0.04,0.024,0.003,0.01,0.082,0.121],[0.107,0.047,0.048,0.076,0.351,0.058,0.023,0.019,0.021,0.012,0.002,0.007,0.06,0.169],[0.176,0.033,0.036,0.015,0.101,0.04,0.043,0.024,0.042,0.019,0.003,0.013,0.099,0.358],[0.092,0.018,0.003,0.001,0.021,0.014,0.013,0.008,0.387,0.098,0.008,0.021,0.157,0.16],[0.042,0.014,0.002,0.0,0.007,0.041,0.033,0.017,0.214,0.063,0.022,0.025,0.271,0.248],[0.048,0.015,0.001,0.0,0.008,0.045,0.085,0.015,0.066,0.02,0.032,0.017,0.166,0.481],[0.086,0.014,0.002,0.0,0.011,0.039,0.049,0.008,0.053,0.028,0.042,0.033,0.107,0.529],[0.114,0.021,0.011,0.002,0.016,0.027,0.026,0.006,0.018,0.022,0.02,0.026,0.174,0.517],[0.141,0.016,0.017,0.004,0.009,0.019,0.011,0.005,0.022,0.018,0.004,0.011,0.141,0.581],[0.169,0.008,0.003,0.001,0.001,0.007,0.005,0.004,0.005,0.005,0.001,0.005,0.055,0.731],[0.165,0.014,0.006,0.001,0.005,0.012,0.008,0.004,0.014,0.011,0.002,0.006,0.066,0.689],[0.221,0.049,0.007,0.002,0.034,0.026,0.012,0.003,0.039,0.014,0.001,0.003,0.092,0.496],[0.203,0.044,0.007,0.004,0.028,0.015,0.012,0.003,0.026,0.008,0.001,0.003,0.06,0.586]],"L1H3":[[0.949,0.005,0.002,0.004,0.002,0.004,0.002,0.002,0.005,0.003,0.003,0.002,0.009,0.009],[0.987,0.01,0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001],[0.861,0.111,0.008,0.004,0.001,0.001,0.0,0.0,0.005,0.002,0.001,0.002,0.002,0.002],[0.349,0.42,0.165,0.008,0.018,0.008,0.002,0.001,0.007,0.004,0.004,0.002,0.006,0.004],[0.472,0.339,0.06,0.037,0.027,0.036,0.001,0.002,0.003,0.001,0.002,0.001,0.013,0.006],[0.382,0.056,0.039,0.053,0.393,0.056,0.004,0.001,0.001,0.002,0.002,0.001,0.006,0.005],[0.084,0.02,0.005,0.012,0.699,0.095,0.041,0.011,0.009,0.003,0.003,0.001,0.008,0.008],[0.067,0.025,0.009,0.004,0.127,0.287,0.416,0.038,0.01,0.004,0.001,0.0,0.003,0.007],[0.341,0.03,0.01,0.012,0.138,0.077,0.038,0.111,0.158,0.036,0.004,0.002,0.014,0.03],[0.438,0.01,0.005,0.002,0.025,0.015,0.015,0.081,0.308,0.076,0.005,0.005,0.006,0.01],[0.276,0.003,0.002,0.003,0.016,0.001,0.008,0.026,0.377,0.213,0.029,0.034,0.01,0.003],[0.183,0.003,0.001,0.001,0.003,0.013,0.014,0.045,0.168,0.113,0.357,0.019,0.059,0.021],[0.353,0.005,0.004,0.002,0.007,0.014,0.004,0.006,0.054,0.155,0.156,0.057,0.124,0.058],[0.895,0.0,0.0,0.0,0.0,0.001,0.0,0.0,0.002,0.003,0.007,0.008,0.047,0.037]],"L6H8":[[0.098,0.099,0.066,0.048,0.094,0.069,0.084,0.053,0.055,0.024,0.027,0.048,0.114,0.122],[0.072,0.13,0.132,0.15,0.087,0.105,0.042,0.021,0.04,0.035,0.019,0.064,0.057,0.047],[0.187,0.109,0.09,0.097,0.067,0.034,0.058,0.014,0.023,0.014,0.023,0.044,0.071,0.169],[0.198,0.108,0.089,0.053,0.072,0.061,0.077,0.018,0.029,0.019,0.016,0.032,0.068,0.16],[0.074,0.1,0.104,0.056,0.064,0.114,0.067,0.097,0.084,0.073,0.016,0.05,0.032,0.07],[0.094,0.128,0.114,0.105,0.058,0.102,0.064,0.064,0.052,0.043,0.021,0.067,0.035,0.054],[0.444,0.025,0.05,0.042,0.012,0.016,0.029,0.035,0.021,0.017,0.015,0.021,0.039,0.236],[0.169,0.054,0.09,0.058,0.019,0.056,0.063,0.032,0.042,0.042,0.035,0.053,0.059,0.228],[0.117,0.119,0.04,0.042,0.11,0.084,0.043,0.066,0.034,0.028,0.023,0.068,0.05,0.177],[0.101,0.188,0.076,0.056,0.12,0.109,0.049,0.033,0.019,0.028,0.03,0.074,0.041,0.076],[0.446,0.063,0.071,0.055,0.032,0.014,0.022,0.036,0.01,0.013,0.013,0.018,0.059,0.149],[0.211,0.148,0.092,0.073,0.069,0.041,0.041,0.046,0.013,0.02,0.045,0.037,0.071,0.094],[0.191,0.187,0.08,0.087,0.168,0.054,0.035,0.035,0.022,0.013,0.01,0.038,0.041,0.038],[0.243,0.028,0.017,0.02,0.029,0.029,0.039,0.021,0.018,0.013,0.009,0.021,0.051,0.462]]}}]}
# DATA-END

BLUE, DARK, YELLOW, ORANGE, GRAY = "#4B8BBE", "#306998", "#FFD43B", "#e65100", "#9ca3af"
SPECIAL = {"[CLS]", "[SEP]"}
SENTENCES = DATA["sentences"]
HEADS = DATA["heads"]

ui.tags.style("""
body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif}
.card{border:1px solid #e5e7eb;border-radius:10px;padding:10px 14px;margin:10px 0;background:#fff}
.muted{color:#6b7280;font-size:.85em}
.grid{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin:8px 0}
.hcard{border:1.5px solid #e5e7eb;border-radius:10px;padding:8px;cursor:pointer;background:#fff;transition:border-color .15s}
.hcard:hover{border-color:#4B8BBE}
.hcard.on{border-color:#306998;box-shadow:0 0 0 3px #30699833}
.hcard .t{font-weight:600;font-size:.85em;line-height:1.2;min-height:2.4em}
.hcard .to{font-size:.8em;margin-top:4px}
.tok:hover rect{stroke:#306998;stroke-width:2.2}
.mat{width:auto;border-collapse:collapse;margin:0 auto}
.mat td{width:26px;height:22px;text-align:center;font-size:.68em;border:1px solid #fff;cursor:pointer}
.mat th{font-weight:500;font-size:.75em;padding:0 4px}
.mat th.col{writing-mode:vertical-rl;transform:rotate(180deg);height:64px;vertical-align:bottom}
@media (max-width:620px){.grid{grid-template-columns:repeat(2,1fr)}}
""")

current = reactive.value(None)
head = reactive.value(0)


@reactive.effect
def _reset():
    current.set(SENTENCES[int(input.sent())]["default"])


@reactive.effect
@reactive.event(input.tok)
def _click():
    current.set(int(input.tok()))


@reactive.effect
@reactive.event(input.head)
def _head():
    head.set(int(input.head()))


def layout(tokens):
    xs, x = [], 6
    for t in tokens:
        w = 9 * len(t) + 18
        xs.append((x, w))
        x += w + 5
    return xs, x + 2


def top_word(tokens, weights, sel):
    """Token de la frase (sin [CLS]/[SEP] ni él mismo) al que más atiende, y su cuota."""
    words = [(w, j) for j, w in enumerate(weights) if j != sel and tokens[j] not in SPECIAL]
    rest = sum(w for w, _ in words) or 1.0
    w_top, j_top = max(words)
    special = sum(w for j, w in enumerate(weights) if tokens[j] in SPECIAL)
    return j_top, w_top / rest, special


def thumb(tokens, att, sel):
    """Huella de la cabeza: matriz completa en miniatura, con la fila seleccionada marcada."""
    n, c = len(tokens), 8
    cells = []
    for i in range(n):
        for j in range(n):
            v = att[i][j]
            if v >= 0.02:
                cells.append(
                    f"<rect x='{j * c}' y='{i * c}' width='{c}' height='{c}' fill='{DARK}' fill-opacity='{min(1, v * 1.4):.2f}'/>"
                )
    sp = "".join(
        f"<rect x='{j * c}' y='0' width='{c}' height='{n * c}' fill='#9ca3af' fill-opacity='.12'/>"
        for j, t in enumerate(tokens) if t in SPECIAL
    )
    row = (
        f"<rect x='-1' y='{sel * c - 1}' width='{n * c + 2}' height='{c + 2}' fill='none' "
        f"stroke='{ORANGE}' stroke-width='1.6'/>"
    )
    return (
        f"<svg viewBox='-2 -2 {n * c + 4} {n * c + 4}' width='100%' style='max-width:130px;display:block;margin:4px auto'>"
        f"<rect x='0' y='0' width='{n * c}' height='{n * c}' fill='#f9fafb'/>{sp}{''.join(cells)}{row}</svg>"
    )


def arcs(tokens, sel, weights):
    xs, total_w = layout(tokens)
    base_y, h = 74, 32
    cx = lambda i: xs[i][0] + xs[i][1] / 2
    parts, labels, placed, cands = [], [], [], []
    for j in sorted(range(len(tokens)), key=lambda j: weights[j]):
        w = weights[j]
        if j == sel or w < 0.03:
            continue
        dist = abs(cx(j) - cx(sel))
        top = base_y - min(52, 12 + dist * 0.12) - 14
        mx = (cx(sel) + cx(j)) / 2
        parts.append(
            f"<path d='M{cx(sel)},{base_y} Q{mx},{top} {cx(j)},{base_y}' "
            f"fill='none' stroke='{DARK}' stroke-width='{1 + 9 * w:.1f}' "
            f"stroke-opacity='{0.25 + 0.75 * w:.2f}' stroke-linecap='round'/>"
        )
        cands.append((w, mx, (base_y + top) / 2))
    for w, mx, apex in sorted(cands, reverse=True):
        if w >= 0.15 and all(abs(mx - px) > 36 or abs(apex - py) > 12 for px, py in placed):
            placed.append((mx, apex))
            labels.append(
                f"<text x='{mx}' y='{apex - 4}' text-anchor='middle' font-size='11' font-weight='600' "
                f"fill='{DARK}' stroke='#fff' stroke-width='3.5' paint-order='stroke'>{w:.0%}</text>"
            )
    for i, t in enumerate(tokens):
        x, wd = xs[i]
        special = t in SPECIAL
        italic = ' font-style="italic"' if special else ""
        if i == sel:
            fill, fop, tcol = DARK, 1, "#fff"
        elif weights[i] >= 0.03:
            fill, fop, tcol = YELLOW, 0.3 + 0.7 * weights[i], "#000"
        else:
            fill, fop, tcol = "none", 1, "currentColor"
        parts.append(
            f"<g class='tok' style='cursor:pointer' opacity='{0.55 if special and i != sel else 1}' "
            f"onclick=\"Shiny.setInputValue('tok', {i}, {{priority: 'event'}})\">"
            f"<rect x='{x}' y='{base_y}' width='{wd}' height='{h}' rx='6' fill='{fill}' "
            f"fill-opacity='{fop}' stroke='{DARK if i == sel else GRAY}' stroke-width='1.2'/>"
            f"<text x='{x + wd / 2}' y='{base_y + 21}' text-anchor='middle' font-size='14' "
            f"fill='{tcol}'{italic}>{t}</text></g>"
        )
    return (
        f"<svg viewBox='0 0 {total_w} {base_y + h + 4}' width='100%' "
        f"style='max-width:{max(total_w, 420)}px;display:block' "
        f"font-family='system-ui,sans-serif'>{''.join(parts)}{''.join(labels)}</svg>"
    )


ui.input_radio_buttons(
    "sent", "Frase",
    {str(i): s["text"] for i, s in enumerate(SENTENCES)}, selected="0", width="100%",
)


@render.ui
def heads_grid():
    s = SENTENCES[int(input.sent())]
    tokens, sel, k = s["tokens"], current(), head()
    if sel is None or sel >= len(tokens):
        return ui.HTML("")
    cards = []
    for h, hd in enumerate(HEADS):
        att = s["att"][hd["id"]]
        j_top, share, special = top_word(tokens, att[sel], sel)
        extra = f" <span class='muted'>· [CLS]/[SEP] {special:.0%}</span>" if special >= 0.25 else ""
        cards.append(
            f"<div class='hcard{' on' if h == k else ''}' "
            f"onclick=\"Shiny.setInputValue('head', {h}, {{priority: 'event'}})\">"
            f"<div class='t'>{hd['label']}</div>{thumb(tokens, att, sel)}"
            f"<div class='to'>«{tokens[sel]}» → <b>«{tokens[j_top]}»</b>{extra}</div></div>"
        )
    return ui.HTML(
        f"<div class='muted' style='margin-top:4px'>Cada tarjeta es la <b>huella</b> de una cabeza: "
        f"la fila <i>i</i> muestra a qué tokens (columnas) atiende el token <i>i</i>. "
        f"La fila naranja es «<b>{tokens[sel]}</b>». Pulsa una tarjeta para ampliarla.</div>"
        f"<div class='grid'>{''.join(cards)}</div>"
    )


@render.ui
def detail():
    s = SENTENCES[int(input.sent())]
    tokens, sel = s["tokens"], current()
    if sel is None or sel >= len(tokens):
        return ui.HTML("")
    hd = HEADS[head()]
    weights = s["att"][hd["id"]][sel]
    j_top, share, special = top_word(tokens, weights, sel)
    anchor = (
        f" <span class='muted'>[CLS]/[SEP] se llevan el {special:.0%} del total.</span>"
        if special >= 0.25 else ""
    )
    return ui.HTML(
        f"<div class='card'><div><b>{hd['label']}</b> "
        f"<span class='muted'>capa {hd['layer'] + 1} · cabeza {hd['head'] + 1}</span></div>"
        f"<div class='muted'>{hd['desc']}</div>"
        f"{arcs(tokens, sel, weights)}"
        f"<div style='font-size:.9em'>Entre las palabras de la frase, «<b>{tokens[sel]}</b>» "
        f"atiende sobre todo a «<b>{tokens[j_top]}</b>» ({share:.0%} de esa parte).{anchor}</div>"
        f"<div class='muted'>Pulsa cualquier token para seleccionarlo.</div></div>"
    )


ui.input_checkbox("nums", "Ver los números de esta cabeza (matriz completa)", value=False, width="100%")


@render.ui
def matrix():
    if not input.nums():
        return ui.HTML("")
    s = SENTENCES[int(input.sent())]
    tokens, sel = s["tokens"], current()
    att = s["att"][HEADS[head()]["id"]]
    th = "".join(f"<th class='col'>{t}</th>" for t in tokens)
    rows = []
    for i, t in enumerate(tokens):
        cells = "".join(
            f"<td style='background:rgba(48,105,152,{min(1, v * 1.4):.2f});color:{'#fff' if v > 0.4 else '#111'}'>"
            f"{v * 100:.0f}</td>" if v >= 0.01 else "<td></td>"
            for v in att[i]
        )
        style = f"outline:2px solid {ORANGE};outline-offset:-1px" if i == sel else ""
        rows.append(
            f"<tr style='{style}' onclick=\"Shiny.setInputValue('tok', {i}, {{priority: 'event'}})\">"
            f"<th style='text-align:right'>{t}</th>{cells}</tr>"
        )
    return ui.HTML(
        f"<div class='card' style='overflow-x:auto'><table class='mat'><thead><tr><th></th>{th}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table>"
        f"<p class='muted' style='margin:6px 0 0'>Porcentaje de atención de cada fila a cada columna "
        f"(cada fila suma 100 %; se ocultan los valores menores del 1 %).</p></div>"
    )
```

## Qué observar

1.  **Empieza por el sujeto** (viene preseleccionado: «perro»,
    «profesora»…) y compara las seis cabezas. La primera mira a la
    palabra de la izquierda, la segunda a la de la derecha y la tercera
    casi no mira a las palabras: se ancla a `[CLS]` y `[SEP]`. Son tres
    cabezas distintas leyendo la misma frase al mismo tiempo.
2.  **Mira la cuarta cabeza:** entre las palabras de la frase tiende a
    saltar hasta el verbo, aunque haya palabras por medio. Pulsa un
    **objeto** («gato», «lección») y mira la quinta, que hace algo
    parecido. Fíjate también en cuánta atención se llevan `[CLS]` y
    `[SEP]`: es una parte grande.
3.  **Compara con la sexta cabeza**, que reparte la atención sin un
    patrón que se pueda describir en una frase. No todas las cabezas
    hacen algo que un humano sepa nombrar.
4.  Fíjate en que algunas palabras se parten en trozos (`persigui` +
    `##ó`). Eso son los **tokens**, no las palabras. El modelo trabaja
    con ellos, como en [la demo de BPE](bpe.qmd).

## Qué has visto

- En un modelo real **no hay una atención**, hay muchas cabezas a la
  vez, y cada una se fija en cosas distintas. El modelo combina todas
  esas «lecturas» de la misma frase.
- Algunas cabezas son muy simples (mirar al vecino) y otras parecen
  seguir la gramática. Otras no tienen un patrón interpretable.
- Esto encaja con lo que describe la literatura sobre BERT
  \[@clark2019bert\]: hay cabezas posicionales, cabezas que se anclan a
  los marcadores especiales y algunas con relaciones sintácticas.

> **Cuidado con lo que no demuestra esta demo**
>
> - **Las cabezas de sujeto→verbo y objeto→verbo son una tendencia, no
>   una regla.** Se eligieron porque en estas cuatro frases (con muy
>   pocos casos) se comportan así, y puede que no lo hagan en otras. Las
>   otras tres (anterior, siguiente y `[CLS]`/`[SEP]`) sí son muy
>   estables.
> - **Aquí no aparece que el modelo «entienda» el significado.** En las
>   cabezas mostradas no se ve nada parecido a distinguir los dos
>   sentidos de *banco* o a resolver un pronombre: eso ocurre repartido
>   a lo largo de las capas y no se lee en una sola cabeza. Por eso la
>   [primera parte](atencion.qmd) es una ilustración.
> - **Atención no equivale a explicación**: que una cabeza atienda a una
>   palabra no demuestra que esa palabra sea la causa de lo que decide
>   el modelo \[@jain2019attention\].

# References